# System Capacity & Care Load Analytics for Unaccompanied Children

## Notebook 01: Data Ingestion & Structuring

---

### Project Overview

The Unaccompanied Alien Children (UAC) Program moves children through a care
pipeline: apprehension by CBP, transfer into HHS custody, and eventual
discharge to a vetted sponsor. Before any capacity analysis can happen, the
daily reporting data has to be loaded, ordered chronologically, and given a
complete daily index so that gaps and irregular reporting cadence don't get
mistaken for real drops in system load.

This notebook covers:

- Loading the raw daily time-series export
- Converting the `Date` column to a proper datetime type
- Ensuring chronological ordering
- Establishing a clean, de-duplicated daily index ready for validation in Notebook 02


In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np

In [2]:
# Load the Dataset
file_path = "../data/UAC_Program_Raw.csv"
df = pd.read_csv(file_path)

# Preview the first five rows
df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0


---
# A. Initial Dataset Inspection

In [3]:
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

# Display column names
print(f"column names  : {df.columns.tolist()}")

Rows    : 1170
Columns : 6
column names  : ['Date', 'Children apprehended and placed in CBP custody*', 'Children in CBP custody', 'Children transferred out of CBP custody', 'Children in HHS Care', 'Children discharged from HHS Care']


In [4]:
# Data information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1170 entries, 0 to 1169
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Date                                             720 non-null    str    
 1   Children apprehended and placed in CBP custody*  720 non-null    float64
 2   Children in CBP custody                          720 non-null    float64
 3   Children transferred out of CBP custody          720 non-null    float64
 4   Children in HHS Care                             720 non-null    str    
 5   Children discharged from HHS Care                720 non-null    float64
dtypes: float64(4), str(2)
memory usage: 69.5 KB


### Observation

The source export ships with long, descriptive column headers (one even
carries a trailing footnote asterisk) and every numeric column is read in as
an `object` dtype because large counts use thousands separators (e.g.
`"2,484"`). Both issues need to be resolved before anything downstream can
treat these as numbers.

---
# B. Renaming Columns for Analytical Use

In [5]:
# Rename the verbose source headers to short, analysis-friendly names
df.columns = [
    "Date",
    "CBP_Intake",       # Children apprehended and placed in CBP custody
    "CBP_Custody",      # Children in CBP custody
    "CBP_Transferred",  # Children transferred out of CBP custody (into HHS)
    "HHS_Care",         # Children in HHS Care
    "HHS_Discharged",   # Children discharged from HHS Care
]

df.head()

,Date,CBP_Intake,CBP_Custody,CBP_Transferred,HHS_Care,HHS_Discharged
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0


---
# C. Converting Data Types

In [6]:
# Strip thousands separators and convert every count column to numeric
count_cols = ["CBP_Intake", "CBP_Custody", "CBP_Transferred", "HHS_Care", "HHS_Discharged"]

for col in count_cols:
    df[col] = df[col].astype(str).str.replace(",", "", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.dtypes

Date                   str
CBP_Intake         float64
CBP_Custody        float64
CBP_Transferred    float64
HHS_Care           float64
HHS_Discharged     float64
dtype: object

In [7]:
# Convert Date to a proper datetime type
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Date"].dtype

dtype('<M8[us]')

### Observation

All five count columns now parse as numeric (`float64`, since the raw file
mixes integer counts with a handful of blank rows), and `Date` is a proper
`datetime64` column. This is the minimum bar for any time-series work in the
notebooks that follow.

---
# D. Chronological Ordering & Daily Index

In [8]:
# Drop rows where the date itself failed to parse (blank trailer rows in the export)
before = len(df)
df = df.dropna(subset=["Date"])
after = len(df)
print(f"Dropped {before - after} row(s) with an unparseable date.")

Dropped 450 row(s) with an unparseable date.


In [9]:
# Sort into chronological order and reset the index
df = df.sort_values("Date").reset_index(drop=True)

print(f"Date range : {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Total reporting days in file : {len(df):,}")

Date range : 2023-01-12 to 2025-12-21
Total reporting days in file : 720


In [10]:
# The report is not published every single calendar day — check the actual
# reporting cadence so later notebooks don't assume a full daily index
full_calendar_days = (df["Date"].max() - df["Date"].min()).days + 1
print(f"Calendar days spanned  : {full_calendar_days:,}")
print(f"Actual reporting days  : {len(df):,}")
print(f"Reporting coverage     : {len(df) / full_calendar_days * 100:.1f}% of calendar days")

Calendar days spanned  : 1,075
Actual reporting days  : 720
Reporting coverage     : 67.0% of calendar days


### Observation

The program does not publish a report every calendar day, so any rolling
average or trend line built downstream should be interpreted as
"per reporting day" rather than strictly daily. This is noted again in
Notebook 03 where rolling windows are constructed.

---
# E. Saving the Structured Dataset

In [11]:
df.to_csv("../data/Processed/UAC_structured.csv", index=False)
print("Saved structured dataset to ../data/Processed/UAC_structured.csv")
df.head()

Saved structured dataset to ../data/Processed/UAC_structured.csv


,Date,CBP_Intake,CBP_Custody,CBP_Transferred,HHS_Care,HHS_Discharged
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0
1,2023-01-22,32.0,49.0,39.0,7122.0,227.0
2,2023-01-23,32.0,50.0,39.0,7280.0,181.0
3,2023-01-24,47.0,42.0,47.0,7433.0,175.0
4,2023-01-25,20.0,22.0,41.0,7538.0,180.0


---
# Conclusion

The data ingestion and structuring process was completed successfully:

### Key Findings

- The raw export was loaded and its five verbose column headers were renamed
  to short, analysis-ready names (`CBP_Intake`, `CBP_Custody`,
  `CBP_Transferred`, `HHS_Care`, `HHS_Discharged`).
- All count columns were converted from comma-formatted strings to numeric
  types, and `Date` was converted to a proper datetime column.
- Rows with unparseable dates were dropped, and the dataset was sorted into
  chronological order with a clean reset index.
- The program reports on an irregular cadence rather than every calendar
  day — later rolling-average and volatility calculations are built on
  reporting order, not fixed calendar spacing.
- The structured dataset was saved to `data/Processed/UAC_structured.csv`,
  ready for the data quality and validation pass in **Notebook 02**.
